# Build the human-rating sample
Code that generates random sample for human verification


In [ ]:
# set reproducibility
RANDOM = 10182026

# set number per model/context API to verify 
N_per_api = 5

In [ ]:

import json
import os

import pandas as pd

INPUTS = [
   "../parsed_output/parsed_anthropic_batch_results.json",
   "../parsed_output/parsed_openai_batch_results_gpt-5-6-so.json",
   "../parsed_output/parsed_openai_batch_results_gpt-5-5-20.json",
   "../parsed_output/parsed_chatgpt_results.json",
   "../parsed_output/parsed_claude_results.json"]

def load_parsed_results(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        df = pd.DataFrame(json.load(f))
    return df

def load_and_concat(paths: list[str]) -> pd.DataFrame:
    return pd.concat(
        [load_parsed_results(p) for p in paths],
        ignore_index=True,
    )
all_df = load_and_concat(INPUTS)


## First keep all chat and claude

In [4]:
all_chats = all_df[(all_df.model=='claude') |(all_df.model=='chatgpt')]
print(f'size of all chatbot output: {len(all_chats)}')

size of all chatbot output: 100


### Random subset of 120 

In [5]:
all_api = all_df[(all_df.model!='claude') & (all_df.model!='chatgpt')]
print(f'size of all api output: {len(all_api)}')

size of all api output: 2880


In [6]:
## randomly select 5 rows by context label and model
sampled_df = (
    all_api.groupby(["prompt_label", "model"], group_keys=False)
      .sample(n=N_per_api, replace=False, random_state=RANDOM)
)

In [7]:
## put back together
df_combined = pd.concat([all_chats, sampled_df], ignore_index=False)

## Check counts

In [8]:
df_combined.groupby(['model','prompt_label']).count()

image_label  prompt_base  prompt  iter_num  \
model      prompt_label                                                   
chatgpt    no_context                  0           50      50        50   
claude     no_context                  0           50      50        50   
claude-opu context_ins_meal            5            5       5         5   
           context_insulin             5            5       5         5   
           context_no_ins              5            5       5         5   
           no_context                  5            5       5         5   
           no_context_mdi              5            5       5         5   
           no_context_pump             5            5       5         5   
claude-son context_ins_meal            5            5       5         5   
           context_insulin             5            5       5         5   
           context_no_ins              5            5       5         5   
           no_context                  5            5       5         5   
           no_context_mdi              5            5       5         5   
           no_context_pump             5            5       5         5   
gpt-5-5-20 context_ins_meal            5            5       5         5   
           context_insulin             5            5       5         5   
           context_no_ins              5            5       5         5   
           no_context                  5            5       5         5   
           no_context_mdi              5            5       5         5   
           no_context_pump             5            5       5         5   
gpt-5-6-so context_ins_meal            5            5       5         5   
           context_insulin             5            5       5         5   
           context_no_ins              5            5       5         5   
           no_context                  5            5       5         5   
           no_context_mdi              5            5       5         5   
           no_context_pump             5            5       5         5   

                             raw_response  parse_status  \
model      prompt_label                                   
chatgpt    no_context                  50            50   
claude     no_context                  50            50   
claude-opu context_ins_meal             5             5   
           context_insulin              5             5   
           context_no_ins               5             5   
           no_context                   5             5   
           no_context_mdi               5             5   
           no_context_pump              5             5   
claude-son context_ins_meal             5             5   
           context_insulin              5             5   
           context_no_ins               5             5   
           no_context                   5             5   
           no_context_mdi               5             5   
           no_context_pump              5             5   
gpt-5-5-20 context_ins_meal             5             5   
           context_insulin              5             5   
           context_no_ins               5             5   
           no_context                   5             5   
           no_context_mdi               5             5   
           no_context_pump              5             5   
gpt-5-6-so context_ins_meal             5             5   
           context_insulin              5             5   
           context_no_ins               5             5   
           no_context                   5             5   
           no_context_mdi               5             5   
           no_context_pump              5             5   

                             take_correction_bolus  \
model      prompt_label                              
chatgpt    no_context                           50   
claude     no_context                           50   
claude-opu context_ins_meal                      5   
     

## Check distributions

In [9]:
df_combined['prompt_label'].value_counts()

prompt_label
no_context          120
context_ins_meal     20
context_insulin      20
context_no_ins       20
no_context_mdi       20
no_context_pump      20
Name: count, dtype: int64

# Shuffle and make spreadsheet

In [10]:
# shuffled
df_shuffled = df_combined.sample(frac=1, random_state=RANDOM).reset_index()

In [11]:
# create ID
df_shuffled = df_shuffled.reset_index(names="ID")
## Save df_shuffled
df_shuffled.to_csv('shuffled_sample_for_annotations.csv')
# backup
df_shuffled.to_pickle("shuffled_sample_for_annotations.pkl")

#drop columns
to_annotate = df_shuffled[['ID','raw_response']]


#### Save annotation CSV

In [12]:
## Save copy for GUI 

to_annotate.to_csv('GUI_code/annotation_gui.csv')
# backup
to_annotate.to_pickle("annotation_gui.pkl")

In [13]:
# add empty columns
# set tasks 
TASK1 = "take_correction_bolus (Options: YES, NO)"
TASK2 = "warns_against_stacking (Options: YES, NO, NA)"
TASK3 = "timing_of_insulin_stacking (Options: IMMEDIATE INSULIN STACKING, FUTURE INSULIN STACKING, AMBIGUOUS, BOTH, NA)"

for col in [TASK1, TASK2, TASK3]:
    to_annotate[col] = None

## Save annotation
to_annotate.to_csv('annotation.csv')
# backup
to_annotate.to_pickle("annotation.pkl")